# Метод 4 — Set Transformer. Часть 1: данные и архитектура

Set Transformer (Lee et al. 2019) с дистрибутивной головой и tail-cumsum
для монотонной по построению reach-кривой `P(X ≥ 1), P(X ≥ 2), P(X ≥ 3)`.


## Пути и константы

In [8]:
from pathlib import Path
import sys

ROOT = Path('/Users/ensamsanovich/auc_forecast')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUT_DIR = ROOT / 'outputs' / 'method_4_set_transformer'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Корневой путь:', ROOT)
print('Папка для PNG:', OUT_DIR)

Корневой путь: /Users/ensamsanovich/auc_forecast
Папка для PNG: /Users/ensamsanovich/auc_forecast/outputs/method_4_set_transformer


## Загрузка единого сплита

In [9]:
from core.config import SEED, N_PUBLISHERS, TARGETS, set_seed
from core.runner import load_unified_data

set_seed(SEED)
data = load_unified_data()
split = data['split']
print(f"Train: {len(split['train_idx'])} | Holdout: {len(split['holdout_idx'])}")
print(f"Число CV-фолдов: {len(split['folds'])}")
print(f"Протокол сплита: {split['protocol']}")

Train: 806 | Holdout: 202
Число CV-фолдов: 5
Протокол сплита: time-based 80/20 (sort by hour_start with row_index tiebreaker) + 5-fold campaign CV on train (seed=42)


In [ ]:
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

from core.config import N_PUBLISHERS, TARGETS
from core.leak_safe_features import UserHistoryIndex


CAMPAIGN_CONT_COLS = ["cpm", "hour_start", "duration", "audience_size", "n_publishers"]
N_USER_FEATURES = 9  # 4 history-std + 1 cold flag + 4 demographic (sex/age/city/unknown)


def precompute_campaign_features(val_df: pd.DataFrame) -> pd.DataFrame:
    """Add `duration` and `n_publishers` columns without leaking future info.

    Accepts either the parsed form (lists in `publishers`/`user_ids`, what
    `core.runner.load_unified_data` produces) or the raw string form.
    """
    df = val_df.copy()
    df["duration"] = df["hour_end"] - df["hour_start"]
    pubs = df["publishers"]
    if len(pubs) > 0 and isinstance(pubs.iloc[0], (list, tuple)):
        df["n_publishers"] = pubs.apply(len).astype(np.int32)
    else:
        df["n_publishers"] = pubs.astype(str).str.split(",").str.len().astype(np.int32)
    return df


def _as_user_id_array(value) -> np.ndarray:
    if isinstance(value, (list, tuple, np.ndarray)):
        return np.asarray(value, dtype=np.int32)
    return np.array([int(x) for x in str(value).split(",") if x], dtype=np.int32)


def _publisher_multihot(value, n_pubs: int = N_PUBLISHERS) -> np.ndarray:
    out = np.zeros(n_pubs, dtype=np.float32)
    iterable = value if isinstance(value, (list, tuple, np.ndarray)) else str(value).split(",")
    for p in iterable:
        idx = int(p) - 1  # publishers are 1-indexed in raw data
        if 0 <= idx < n_pubs:
            out[idx] = 1.0
    return out


@dataclass
class _DemoLookup:
    sex: np.ndarray
    age: np.ndarray
    city: np.ndarray
    known: np.ndarray
    max_uid: int

    @classmethod
    def build(cls, users_df: pd.DataFrame, age_max: int = 120, city_max: int = 3000) -> "_DemoLookup":
        u = users_df.set_index("user_id")
        max_uid = int(u.index.max())
        sex = np.zeros(max_uid + 2, dtype=np.float32)
        age = np.zeros(max_uid + 2, dtype=np.float32)
        city = np.zeros(max_uid + 2, dtype=np.float32)
        known = np.zeros(max_uid + 2, dtype=np.float32)
        for uid, r in u.iterrows():
            sex[uid] = float(r["sex"]) / 2.0
            age[uid] = float(r["age"]) / age_max
            city[uid] = float(r["city_id"]) / city_max
            known[uid] = 1.0
        return cls(sex=sex, age=age, city=city, known=known, max_uid=max_uid)

    def demo_block(self, user_ids: np.ndarray) -> np.ndarray:
        """[n_users, 4] -> (sex, age, city, is_unknown_user)."""
        demo = np.zeros((len(user_ids), 4), dtype=np.float32)
        for j, uid in enumerate(user_ids):
            if 0 <= uid <= self.max_uid and self.known[uid] > 0:
                demo[j, 0] = self.sex[uid]
                demo[j, 1] = self.age[uid]
                demo[j, 2] = self.city[uid]
            else:
                demo[j, 3] = 1.0
        return demo


class CampaignSetDataset(Dataset):

    def __init__(
        self,
        campaign_rows: pd.DataFrame,
        users_df: pd.DataFrame,
        hist_index: UserHistoryIndex,
        targets_df: pd.DataFrame | None,
        camp_feat_mean: np.ndarray,
        camp_feat_std: np.ndarray,
        user_feat_mean: np.ndarray,
        user_feat_std: np.ndarray,
        n_publishers: int = N_PUBLISHERS,
    ):
        self.campaigns = campaign_rows.reset_index(drop=True)
        self.targets_df = targets_df.reset_index(drop=True) if targets_df is not None else None
        self.n_publishers = n_publishers
        demo = _DemoLookup.build(users_df)

        self._user_feat_cache: list[torch.Tensor] = []
        self._camp_feat_cache: list[torch.Tensor] = []
        self._pub_cache: list[torch.Tensor] = []
        self._target_cache: list[torch.Tensor] = []
        self._n_users_cache: list[int] = []

        for idx in range(len(self.campaigns)):
            row = self.campaigns.iloc[idx]
            cutoff = int(row["hour_start"])
            user_ids = _as_user_id_array(row["user_ids"])

            hist_feats = np.empty((len(user_ids), 5), dtype=np.float32)
            for j, uid in enumerate(user_ids):
                hist_feats[j] = hist_index.user_features_before(int(uid), cutoff)
            hist_std = (hist_feats[:, :4] - user_feat_mean[:4]) / (user_feat_std[:4] + 1e-6)

            demo_block = demo.demo_block(user_ids)
            user_feats = np.concatenate(
                [hist_std, hist_feats[:, 4:5], demo_block], axis=1
            ).astype(np.float32)
            assert user_feats.shape[1] == N_USER_FEATURES

            camp_raw = np.array([row[c] for c in CAMPAIGN_CONT_COLS], dtype=np.float32)
            camp_feats = ((camp_raw - camp_feat_mean) / (camp_feat_std + 1e-6)).astype(np.float32)
            pub_mh = _publisher_multihot(row["publishers"], n_publishers)

            self._user_feat_cache.append(torch.from_numpy(user_feats))
            self._camp_feat_cache.append(torch.from_numpy(camp_feats))
            self._pub_cache.append(torch.from_numpy(pub_mh))
            self._n_users_cache.append(int(len(user_ids)))
            if self.targets_df is not None:
                self._target_cache.append(torch.tensor(
                    [self.targets_df.loc[idx, t] for t in TARGETS], dtype=torch.float32
                ))

    def __len__(self) -> int:
        return len(self.campaigns)

    def __getitem__(self, idx: int) -> dict:
        out = {
            "user_feats": self._user_feat_cache[idx],
            "camp_feats": self._camp_feat_cache[idx],
            "pub_mh": self._pub_cache[idx],
            "n_users": self._n_users_cache[idx],
        }
        if self.targets_df is not None:
            out["target"] = self._target_cache[idx]
        return out


def collate_pad(batch: list[dict]) -> dict:
    max_n = max(b["n_users"] for b in batch)
    B = len(batch)
    D_user = batch[0]["user_feats"].shape[1]
    user_feats = torch.zeros(B, max_n, D_user)
    mask = torch.zeros(B, max_n, dtype=torch.bool)
    camp_feats = torch.stack([b["camp_feats"] for b in batch])
    pub_mh = torch.stack([b["pub_mh"] for b in batch])
    has_target = "target" in batch[0]
    targets = torch.stack([b["target"] for b in batch]) if has_target else None
    for i, b in enumerate(batch):
        n = b["n_users"]
        user_feats[i, :n] = b["user_feats"]
        mask[i, :n] = True
    return {
        "user_feats": user_feats,
        "mask": mask,
        "camp_feats": camp_feats,
        "pub_mh": pub_mh,
        "target": targets,
    }


def fit_normalizers(
    campaigns: pd.DataFrame,
    train_idx: np.ndarray,
    hist_index: UserHistoryIndex,
    sample_seed: int = 42,
    user_sample_cap: int = 200,
) -> dict:
    train_camps = campaigns.iloc[train_idx]
    camp_X = train_camps[CAMPAIGN_CONT_COLS].to_numpy(dtype=np.float32)
    camp_mean = camp_X.mean(axis=0)
    camp_std = camp_X.std(axis=0) + 1e-6

    rng = np.random.default_rng(sample_seed)
    user_feat_samples = []
    for _, row in train_camps.iterrows():
        cutoff = int(row["hour_start"])
        uids = _as_user_id_array(row["user_ids"])
        if len(uids) > user_sample_cap:
            uids = rng.choice(uids, user_sample_cap, replace=False)
        for uid in uids:
            user_feat_samples.append(hist_index.user_features_before(int(uid), cutoff))
    U = np.stack(user_feat_samples, axis=0)
    user_mean = U[:, :4].mean(axis=0)
    user_std = U[:, :4].std(axis=0) + 1e-6
    return {
        "camp_mean": camp_mean,
        "camp_std": camp_std,
        "user_mean": user_mean,
        "user_std": user_std,
    }


In [ ]:
from __future__ import annotations

import math

import torch
import torch.nn as nn
import torch.nn.functional as F


class MAB(nn.Module):

    def __init__(self, d_q: int, d_k: int, d_out: int, n_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert d_out % n_heads == 0, f"d_out ({d_out}) must be divisible by n_heads ({n_heads})"
        self.n_heads = n_heads
        self.d_head = d_out // n_heads
        self.fc_q = nn.Linear(d_q, d_out)
        self.fc_k = nn.Linear(d_k, d_out)
        self.fc_v = nn.Linear(d_k, d_out)
        self.fc_o = nn.Linear(d_out, d_out)
        self.ln0 = nn.LayerNorm(d_out)
        self.ln1 = nn.LayerNorm(d_out)
        self.ff = nn.Sequential(
            nn.Linear(d_out, d_out * 2), nn.GELU(), nn.Linear(d_out * 2, d_out)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, Q: torch.Tensor, K: torch.Tensor, mask_k: torch.Tensor | None = None) -> torch.Tensor:
        B, m, _ = Q.shape
        n = K.shape[1]
        q = self.fc_q(Q).view(B, m, self.n_heads, self.d_head).transpose(1, 2)
        k = self.fc_k(K).view(B, n, self.n_heads, self.d_head).transpose(1, 2)
        v = self.fc_v(K).view(B, n, self.n_heads, self.d_head).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if mask_k is not None:
            scores = scores.masked_fill(~mask_k[:, None, None, :], float("-inf"))
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, m, self.n_heads * self.d_head)
        out = self.fc_o(out)
        # Project Q via fc_q for residual (so dims match) — equivalent to taking the residual on `q` reshaped.
        q_skip = self.fc_q(Q) if Q.shape[-1] != out.shape[-1] else Q
        h = self.ln0(q_skip + out)
        h = self.ln1(h + self.ff(h))
        return h


class ISAB(nn.Module):

    def __init__(self, d_in: int, d_out: int, m_ind: int = 16, n_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.inducing = nn.Parameter(torch.randn(1, m_ind, d_out) * 0.02)
        self.mab1 = MAB(d_out, d_in, d_out, n_heads, dropout)
        self.mab2 = MAB(d_in, d_out, d_out, n_heads, dropout)

    def forward(self, X: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        B = X.shape[0]
        I = self.inducing.expand(B, -1, -1)
        H = self.mab1(I, X, mask_k=mask)          # [B, m_ind, d_out]
        out = self.mab2(X, H, mask_k=None)         # [B, n, d_out]
        return out


class PMA(nn.Module):
    def __init__(self, d: int, k: int = 1, n_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.seeds = nn.Parameter(torch.randn(1, k, d) * 0.02)
        self.mab = MAB(d, d, d, n_heads, dropout)

    def forward(self, X: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        B = X.shape[0]
        S = self.seeds.expand(B, -1, -1)
        return self.mab(S, X, mask_k=mask)         # [B, k, d]


class SetTransformerReach(nn.Module):

    def __init__(
        self,
        d_user: int,
        d_camp: int,
        n_publishers: int = 21,
        d_hidden: int = 96,
        n_isab: int = 2,
        m_ind: int = 16,
        n_heads: int = 4,
        k_max: int = 6,
        dropout: float = 0.1,
        use_attention: bool = True,
        use_distribution: bool = True,
    ):
        super().__init__()
        self.use_attention = use_attention
        self.use_distribution = use_distribution
        self.k_max = k_max
        self.d_hidden = d_hidden

        self.user_enc = nn.Sequential(
            nn.Linear(d_user, d_hidden),
            nn.GELU(),
            nn.Linear(d_hidden, d_hidden),
        )

        if use_attention:
            self.isab = nn.ModuleList(
                [ISAB(d_hidden, d_hidden, m_ind, n_heads, dropout) for _ in range(n_isab)]
            )
            self.pma = PMA(d_hidden, k=1, n_heads=n_heads, dropout=dropout)
        else:
            self.isab = None
            self.pma = None

        self.camp_enc = nn.Sequential(
            nn.Linear(d_camp + n_publishers, d_hidden),
            nn.GELU(),
            nn.Linear(d_hidden, d_hidden),
        )

        head_in = d_hidden * 2
        head_out = (k_max + 1) if use_distribution else 3
        self.head = nn.Sequential(
            nn.Linear(head_in, d_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, head_out),
        )

    def forward(self, batch: dict) -> dict:
        X = batch["user_feats"]               # [B, n, D_user]
        mask = batch["mask"]                  # [B, n]
        camp = batch["camp_feats"]            # [B, D_camp]
        pub = batch["pub_mh"]                 # [B, n_pubs]

        H = self.user_enc(X)                  # [B, n, d_hidden]
        if self.use_attention:
            for blk in self.isab:
                H = blk(H, mask=mask)
            pooled = self.pma(H, mask=mask).squeeze(1)        # [B, d_hidden]
        else:
            m = mask.unsqueeze(-1).float()
            pooled = (H * m).sum(dim=1) / m.sum(dim=1).clamp(min=1.0)

        camp_repr = self.camp_enc(torch.cat([camp, pub], dim=1))
        joint = torch.cat([pooled, camp_repr], dim=1)
        logits = self.head(joint)

        if self.use_distribution:
            probs = F.softmax(logits, dim=1)                  # [B, k_max+1]
            # tail-cumsum: y_k = P(X >= k) = sum_{m=k..K} probs[m]
            cum_from_top = torch.cumsum(probs.flip(1), dim=1).flip(1)  # [B, k_max+1]
            y_hat = cum_from_top[:, 1:4]                       # [B, 3]
            return {"y_hat": y_hat, "probs": probs}
        else:
            y_hat = torch.sigmoid(logits)                      # ablation: no monotone guarantee
            return {"y_hat": y_hat, "probs": None}


## Sanity-check архитектуры

In [12]:
import torch

model = SetTransformerReach(
    d_user=N_USER_FEATURES,
    d_camp=len(CAMPAIGN_CONT_COLS),
    n_publishers=N_PUBLISHERS,
)
n_params = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Всего параметров: {n_params:,}')
print(f'Обучаемых параметров: {n_train:,}')
print(model)

Всего параметров: 418,471
Обучаемых параметров: 418,471
SetTransformerReach(
  (user_enc): Sequential(
    (0): Linear(in_features=9, out_features=96, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=96, out_features=96, bias=True)
  )
  (isab): ModuleList(
    (0-1): 2 x ISAB(
      (mab1): MAB(
        (fc_q): Linear(in_features=96, out_features=96, bias=True)
        (fc_k): Linear(in_features=96, out_features=96, bias=True)
        (fc_v): Linear(in_features=96, out_features=96, bias=True)
        (fc_o): Linear(in_features=96, out_features=96, bias=True)
        (ln0): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
        (ln1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
        (ff): Sequential(
          (0): Linear(in_features=96, out_features=192, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=192, out_features=96, bias=True)
        )
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (

In [ ]:

from __future__ import annotations

import numpy as np
import pandas as pd

from core.config import HOLDOUT_FRAC, SEED
from core.data_io import load_raw
from core.leak_safe_features import UserHistoryIndex
from core.splits import time_based_split

def _load_campaigns() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    users, history, val, _ = load_raw()
    campaigns = precompute_campaign_features(val)
    return users, history, campaigns


def test_no_future_history_in_user_features(n_check_campaigns: int = 20, seed: int = 1) -> dict:
    users, history, campaigns = _load_campaigns()
    hist_base = UserHistoryIndex.build(history)

    rng = np.random.default_rng(seed)
    rows = rng.choice(len(campaigns), n_check_campaigns, replace=False)
    max_abs_diff = 0.0
    for r in rows:
        row = campaigns.iloc[r]
        cutoff = int(row["hour_start"])
        h2 = history.copy()
        mask = h2["hour"] >= cutoff
        h2.loc[mask, "cpm"] = rng.uniform(1, 999, size=int(mask.sum()))
        h2.loc[mask, "publisher"] = rng.integers(1, 21, size=int(mask.sum()))
        hist_pert = UserHistoryIndex.build(h2)

        uids = _as_user_id_array(row["user_ids"])
        for uid in uids[:30]:
            f_base = hist_base.user_features_before(int(uid), cutoff)
            f_pert = hist_pert.user_features_before(int(uid), cutoff)
            max_abs_diff = max(max_abs_diff, float(np.abs(f_base - f_pert).max()))
    return {
        "max_abs_diff": max_abs_diff,
        "passed": max_abs_diff < 1e-6,
        "checked_campaigns": int(len(rows)),
    }


def test_holdout_split_temporal() -> dict:
    _, _, campaigns = _load_campaigns()
    train_idx, holdout_idx = time_based_split(campaigns, holdout_frac=HOLDOUT_FRAC)
    max_train_hs = int(campaigns.iloc[train_idx]["hour_start"].max())
    min_hold_hs = int(campaigns.iloc[holdout_idx]["hour_start"].min())
    return {
        "max_train_hour_start": max_train_hs,
        "min_holdout_hour_start": min_hold_hs,
        "passed": max_train_hs <= min_hold_hs,
        "n_train": int(len(train_idx)),
        "n_holdout": int(len(holdout_idx)),
    }


def test_normalizers_train_only() -> dict:
    _, history, campaigns = _load_campaigns()
    hist_index = UserHistoryIndex.build(history)
    train_idx, holdout_idx = time_based_split(campaigns, holdout_frac=HOLDOUT_FRAC)

    norm1 = fit_normalizers(campaigns, train_idx, hist_index, sample_seed=SEED)
    campaigns2 = campaigns.copy()
    campaigns2.loc[holdout_idx, "cpm"] = -999.0
    norm2 = fit_normalizers(campaigns2, train_idx, hist_index, sample_seed=SEED)
    same = all(
        np.allclose(norm1[k], norm2[k]) for k in ("camp_mean", "camp_std", "user_mean", "user_std")
    )
    return {"passed": same}


def run_all(n_check_campaigns: int = 20) -> dict:
    results = {
        "no_future_history": test_no_future_history_in_user_features(n_check_campaigns),
        "temporal_holdout": test_holdout_split_temporal(),
        "normalizers_train_only": test_normalizers_train_only(),
    }
    results["all_passed"] = all(r["passed"] for r in results.values() if isinstance(r, dict))
    return results


In [14]:
results = run_all(n_check_campaigns=20)
for k, v in results.items():
    print(k, '→', v)

no_future_history → {'max_abs_diff': 0.0, 'passed': True, 'checked_campaigns': 20}
temporal_holdout → {'max_train_hour_start': 1256, 'min_holdout_hour_start': 1256, 'passed': True, 'n_train': 806, 'n_holdout': 202}
normalizers_train_only → {'passed': True}
all_passed → True
